# Connecticut — CGS Title 38a (Insurance) → `data/connecticut/ins_codes/*.md`

Connecticut insurance law is **Title 38a — Insurance** of the **Connecticut General Statutes (C.G.S.)**.

The General Assembly publishes the current statutes under **`www.cga.ct.gov/current/pub/`**. The title index **`title_38a.htm`** links to each **chapter** (`chap_697.htm`, `chap_700c.htm`, …). Every section is a **`<p>`** whose heading is **`<span class="catchln" id="sec_38a-…">** (or **`secs_38a-…`** for reserved ranges).

This notebook:

1. Loads **`title_38a.htm`** and collects every **`chap_*.htm`** link for Title 38a.
2. Fetches each chapter page, extracts one Markdown file per **`sec_38a-*`** / **`secs_38a-*`** block.

**Official source:** each file cites the chapter URL plus the section **fragment** (`#sec_38a-…`).

**Volume:** on the order of **1,400** sections — full run takes several minutes. Use **`MAX_SECTIONS`** to cap exports while testing.

**SSL:** If you see certificate errors on macOS, set **`VERIFY_SSL = False`** in the config cell (less secure) or ensure **`certifi`** is installed.

**Politeness:** **`REQUEST_DELAY_SEC`** (default **0.25 s**) between chapter requests.

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q httpx beautifulsoup4 certifi


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin

import certifi
import httpx
from bs4 import BeautifulSoup

PUB_BASE = "https://www.cga.ct.gov/current/pub/"
TITLE_38A_PAGE = PUB_BASE + "title_38a.htm"

OUT_DIR = Path("data") / "connecticut" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-CT-CGS-38a/1.0 (public Connecticut General Statutes; educational indexing)"
REQUEST_DELAY_SEC = 0.25
TIMEOUT = 90.0

MAX_SECTIONS = 0
SKIP_EXISTING = True

VERIFY_SSL = True
_verify = certifi.where() if VERIFY_SSL else False

CHAP_HREF = re.compile(r"chap_\d+[a-z]?\.htm$", re.I)


In [3]:
def fetch_text(client: httpx.Client, url: str) -> str:
    r = client.get(url, follow_redirects=True)
    r.raise_for_status()
    time.sleep(REQUEST_DELAY_SEC)
    return r.text


def discover_chapter_urls(client: httpx.Client) -> list[str]:
    html = fetch_text(client, TITLE_38A_PAGE)
    soup = BeautifulSoup(html, "html.parser")
    out: list[str] = []
    seen: set[str] = set()
    for a in soup.find_all("a", href=True):
        h = a["href"]
        if not CHAP_HREF.search(h):
            continue
        absu = urljoin(TITLE_38A_PAGE, h)
        if absu not in seen:
            seen.add(absu)
            out.append(absu)
    return out


def parse_section_paragraphs(html: str, chapter_url: str) -> list[tuple[str, str, str]]:
    """Return [(anchor_id, plain_text, official_url_with_fragment), ...]."""
    soup = BeautifulSoup(html, "html.parser")
    rows: list[tuple[str, str, str]] = []
    for p in soup.find_all("p"):
        span = p.find("span", id=True)
        if not span:
            continue
        cid = span.get("id", "")
        if not (cid.startswith("sec_38a") or cid.startswith("secs_38a")):
            continue
        cls = span.get("class") or []
        if "catchln" not in cls:
            continue
        text = p.get_text("\n", strip=True)
        frag_url = f"{chapter_url}#{cid}"
        rows.append((cid, text, frag_url))
    return rows


def anchor_sort_key(anchor_id: str) -> tuple:
    if anchor_id.startswith("secs_"):
        return (10**9, anchor_id)
    m = re.search(r"38a-(\d+)([a-z]?)", anchor_id, re.I)
    if m:
        return (int(m.group(1)), m.group(2) or "\x00")
    return (10**9, anchor_id)


def anchor_to_filename(anchor_id: str) -> str:
    body = anchor_id
    if body.startswith("sec_"):
        body = body[4:]
    elif body.startswith("secs_"):
        body = body[5:]
    safe = re.sub(r"[^0-9a-zA-Z]+", "_", body).strip("_")
    return f"CGS_sec_{safe}.md"


def anchor_to_section_label(anchor_id: str) -> str:
    if anchor_id.startswith("sec_"):
        return anchor_id[4:]
    if anchor_id.startswith("secs_"):
        return anchor_id[5:]
    return anchor_id


def download_title38a() -> dict[str, int]:
    with httpx.Client(
        headers={"User-Agent": USER_AGENT, "Accept": "text/html,*/*;q=0.8"},
        timeout=TIMEOUT,
        verify=_verify,
        http2=False,
    ) as client:
        chapters = discover_chapter_urls(client)
        print(f"Found {len(chapters)} chapter pages for Title 38a")
        (OUT_DIR / "_connecticut_title38a_chapters.txt").write_text(
            "\n".join(chapters), encoding="utf-8"
        )

        all_rows: list[tuple[str, str, str, tuple]] = []
        for ch_url in chapters:
            h = fetch_text(client, ch_url)
            for aid, body, frag in parse_section_paragraphs(h, ch_url):
                all_rows.append((aid, body, frag, anchor_sort_key(aid)))

    all_rows.sort(key=lambda r: r[3])
    # De-dupe by anchor id (first occurrence wins)
    dedup: dict[str, tuple[str, str]] = {}
    for aid, body, frag, _ in all_rows:
        if aid not in dedup:
            dedup[aid] = (body, frag)

    ordered = sorted(dedup.items(), key=lambda kv: anchor_sort_key(kv[0]))
    print(f"Unique section anchors: {len(ordered)}")

    todo = ordered if not MAX_SECTIONS else ordered[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited export to first {len(todo)} sections (MAX_SECTIONS)")

    wrote, skipped = 0, 0
    for aid, (body, frag) in todo:
        dest = OUT_DIR / anchor_to_filename(aid)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
            continue
        label = anchor_to_section_label(aid)
        title = f"Connecticut General Statutes — Title 38a — § {label}"
        md = (
            f"# {title}\n\n"
            f"**Connecticut General Statutes — Title 38a (Insurance)**\n\n"
            f"**Official source:** {frag}\n\n"
            f"**Section:** {label}\n\n"
            f"**HTML anchor id:** {aid}\n\n"
            f"---\n\n"
            f"{body}\n"
        )
        dest.write_text(md, encoding="utf-8")
        wrote += 1

    print(f"Done. wrote={wrote} skipped={skipped} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped}


download_title38a()


Found 33 chapter pages for Title 38a
Unique section anchors: 1393
Done. wrote=1393 skipped=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/connecticut/ins_codes


{'wrote': 1393, 'skipped': 0}

## Next step

`python -m app.ingest` from the repository root.
